# Cattle_Herd_Performance_Dashboard.ipynb.

In [29]:
# STEP 1: IMPORT LIBRARIES AND CONNECTING TO YOUR GOOGLE SHEETS


# 1. Import libraries
import pandas as pd
import numpy as np

# 2. Authenticate your Google account
from google.colab import auth
auth.authenticate_user()

# 3. Import Google Sheets tools
import google.auth
import gspread

# 2. Authenticate the user's Google account
from google.colab import auth
auth.authenticate_user()

# 3. Connect to Google Sheets
import google.auth
import gspread

credentials, project = google.auth.default()
gc = gspread.authorize(credentials)

# 4. Ask the farmer for their spreadsheet link
spreadsheet_url = input(
    "Paste the link to your Google Sheets workbook: "
).strip()

# 5. Connect to the selected spreadsheet
if not spreadsheet_url:
    raise ValueError("Please enter a Google Sheets link.")

spreadsheet = gc.open_by_url(spreadsheet_url)

# 6. Confirm successful connection
print("\nSuccessfully connected to your farm records!")
print("Spreadsheet:", spreadsheet.title)









Paste the link to your Google Sheets workbook: https://docs.google.com/spreadsheets/d/1Ac7LA-65AcoRxtb1XUFpnVqwNiGiBoGdIY4IcR9czwM/edit?gid=1859157166#gid=1859157166

Successfully connected to your farm records!
Spreadsheet: Cattle Record Spreadsheet 


In [32]:


# STEP 2: CREATE AND LOAD FARM RECORDS

# 1. Define the required spreadsheet tabs and headers

sheet_columns = {

    "Cattle": [
        "Animal_ID",
        "Breed",
        "Date_Acquired",
        "Starting_Weight",
        "Days_On_Feed"
    ],

    "Weight_Records": [
        "Animal_ID",
        "Weight_Date",
        "Weight_Lbs"
    ],

    "Feed_Purchases": [
        "Purchase_ID",
        "Purchase_Date",
        "Feed_Type",
        "Quantity",
        "Unit",
        "Unit_Price",
        "Total_Cost"
    ],

    "Feed_Allocations": [
        "Purchase_ID",
        "Animal_ID",
        "Allocated_Quantity",
        "Allocated_Cost"
    ],

    "Treatment_Records": [
        "Treatment_ID",
        "Animal_ID",
        "Treatment_Date",
        "Treatment_Type",
        "Medication",
        "Cost"
    ],

    "Sales_Processing": [
        "Animal_ID",
        "Sale_Date",
        "Final_Live_Weight",
        "Hanging_Weight",
        "Sale_Basis",
        "Price_Per_Lb"
    ]
}


# 2. Check, create, and prepare each spreadsheet tab

for tab_name, columns in sheet_columns.items():

    try:

        worksheet = spreadsheet.worksheet(tab_name)

        print("Found existing tab:", tab_name)

    except gspread.WorksheetNotFound:

        worksheet = spreadsheet.add_worksheet(
            title=tab_name,
            rows=1000,
            cols=len(columns)
        )

        print("Created missing tab:", tab_name)


    # Get existing spreadsheet information
    existing_values = worksheet.get_all_values()


    # Add headers if the tab is completely empty

    if not existing_values:

        worksheet.update(
            range_name="A1",
            values=[columns]
        )

        print("Added headers:", tab_name)


    else:

        # Remove extra spaces from existing headers
        cleaned_headers = [
            header.strip()
            for header in existing_values[0]
        ]

        # Remove trailing empty header cells
        while cleaned_headers and cleaned_headers[-1] == "":
            cleaned_headers.pop()


        # Save cleaned headers back into Google Sheets
        worksheet.update(
            range_name="A1",
            values=[cleaned_headers]
        )


        # Check that headers match the required format
        if cleaned_headers != columns:

            raise ValueError(
                f"Headers in '{tab_name}' do not match "
                "the required format.\n"
                f"Expected: {columns}\n"
                f"Found: {cleaned_headers}"
            )

        print("Headers verified:", tab_name)


# 3. Load records from Google Sheets

def load_farm_records(tab_name):

    worksheet = spreadsheet.worksheet(tab_name)

    records = worksheet.get_all_records()

    return pd.DataFrame(
        records,
        columns=sheet_columns[tab_name]
    )


# 4. Load all six farm record categories

cattle = load_farm_records("Cattle")

weights = load_farm_records("Weight_Records")

feed_purchases = load_farm_records("Feed_Purchases")

feed_allocations = load_farm_records("Feed_Allocations")

treatments = load_farm_records("Treatment_Records")

sales = load_farm_records("Sales_Processing")


# 5. Display successful loading results

print("\n======================================")
print("FARM RECORDS SUCCESSFULLY LOADED")
print("======================================")

print("Animals:", len(cattle))

print("Weight records:", len(weights))

print("Feed purchases:", len(feed_purchases))

print("Feed allocations:", len(feed_allocations))

print("Treatment records:", len(treatments))

print("Sales records:", len(sales))

print("\nYour cattle tracker is ready!")




Found existing tab: Cattle
Headers verified: Cattle
Found existing tab: Weight_Records
Headers verified: Weight_Records
Found existing tab: Feed_Purchases
Headers verified: Feed_Purchases
Found existing tab: Feed_Allocations
Headers verified: Feed_Allocations
Found existing tab: Treatment_Records
Headers verified: Treatment_Records
Found existing tab: Sales_Processing
Headers verified: Sales_Processing

FARM RECORDS SUCCESSFULLY LOADED
Animals: 0
Weight records: 0
Feed purchases: 0
Feed allocations: 0
Treatment records: 0
Sales records: 0

Your cattle tracker is ready!


In [34]:
#Step 3 Prepare and Clean Farm Records
# 1. Clean Animal IDs
# Keeps animal IDs consistent across spreadsheet tabs.

for df in [
    cattle,
    weights,
    feed_purchases,
    feed_allocations,
    treatments,
    sales
]:

    if "Animal_ID" in df.columns:

        df["Animal_ID"] = (
            df["Animal_ID"]
            .replace("", np.nan)
            .astype("string")
            .str.strip()
        )


# 2. Convert numeric columns into numbers
# Blank or invalid entries become NaN instead of causing errors.

numeric_columns = {

    "cattle": [
        "Starting_Weight",
        "Days_On_Feed"
    ],

    "weights": [
        "Weight_Lbs"
    ],

    "feed_purchases": [
        "Quantity",
        "Unit_Price",
        "Total_Cost"
    ],

    "feed_allocations": [
        "Allocated_Quantity",
        "Allocated_Cost"
    ],

    "treatments": [
        "Cost"
    ],

    "sales": [
        "Final_Live_Weight",
        "Hanging_Weight",
        "Price_Per_Lb"
    ]
}


# Convert numeric information safely

for df_name, columns in numeric_columns.items():

    df = globals()[df_name]

    for column in columns:

        if column in df.columns:

            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            )


# 3. Convert date columns into date format
# Uses Weight_Date to match your Google Sheets headers.

date_columns = {

    "cattle": ["Date_Acquired"],

    "weights": ["Weight_Date"],

    "feed_purchases": ["Purchase_Date"],

    "treatments": ["Treatment_Date"],

    "sales": ["Sale_Date"]
}


# Convert dates safely

for df_name, columns in date_columns.items():

    df = globals()[df_name]

    for column in columns:

        if column in df.columns:

            df[column] = pd.to_datetime(
                df[column],
                errors="coerce"
            )


# 4. Remove completely blank records
# Keeps empty optional tabs usable without counting blank rows.

all_records = {
    "cattle": cattle,
    "weights": weights,
    "feed_purchases": feed_purchases,
    "feed_allocations": feed_allocations,
    "treatments": treatments,
    "sales": sales
}

for df_name, df in all_records.items():

    df.dropna(how="all", inplace=True)


# 5. Display a summary of available farm records

print("Farm records prepared successfully!")

print("\nFARM RECORD SUMMARY")
print("------------------------------")

print("Animals:", len(cattle))
print("Weight records:", len(weights))
print("Feed purchases:", len(feed_purchases))
print("Feed allocations:", len(feed_allocations))
print("Treatment records:", len(treatments))
print("Sales records:", len(sales))


# 6. Identify which optional records have been entered

print("\nOPTIONAL RECORD STATUS")
print("------------------------------")

optional_records = {

    "Feed Purchases": feed_purchases,

    "Feed Allocations": feed_allocations,

    "Treatment Records": treatments,

    "Sales and Processing": sales
}

for record_name, df in optional_records.items():

    if df.empty:

        print(record_name + ": No records entered yet")

    else:

        print(record_name + ": Records available")


# 7. Confirm the tracker is ready for analysis

if cattle.empty:

    print("\nWARNING: No cattle records have been entered yet.")
    print("Add cattle information before running performance analysis.")

else:

    print("\nCattle records are available for analysis.")

print("\nStep 3 completed successfully!")


Farm records prepared successfully!

FARM RECORD SUMMARY
------------------------------
Animals: 0
Weight records: 0
Feed purchases: 0
Feed allocations: 0
Treatment records: 0
Sales records: 0

OPTIONAL RECORD STATUS
------------------------------
Feed Purchases: No records entered yet
Feed Allocations: No records entered yet
Treatment Records: No records entered yet
Sales and Processing: No records entered yet

Add cattle information before running performance analysis.

Step 3 completed successfully!


In [35]:
# STEP 4 CALCULATING CATTLE WEIGHT PERFORMANCE

# 1. Start with the main cattle information

performance = cattle[
    ["Animal_ID", "Breed", "Starting_Weight", "Days_On_Feed"]
].copy()

# Remove records without an Animal ID
performance = performance.dropna(subset=["Animal_ID"])

# Make sure each animal appears only once
performance = performance.drop_duplicates(
    subset=["Animal_ID"],
    keep="last"
)


# 2. Prepare the weight records

# Use Weight_Date to match the Google Sheets column name
valid_weights = weights.dropna(
    subset=["Animal_ID", "Weight_Date", "Weight_Lbs"]
).copy()

# Sort weight records from oldest to newest
valid_weights = valid_weights.sort_values(
    ["Animal_ID", "Weight_Date"]
)


# 3. Identify the first and latest recorded weights

if not valid_weights.empty:

    # Find the first recorded weight for each animal
    first_weights = (
        valid_weights
        .drop_duplicates("Animal_ID", keep="first")
        [["Animal_ID", "Weight_Lbs", "Weight_Date"]]
        .rename(columns={
            "Weight_Lbs": "First_Recorded_Weight",
            "Weight_Date": "First_Weight_Date"
        })
    )

    # Find the latest recorded weight for each animal
    latest_weights = (
        valid_weights
        .drop_duplicates("Animal_ID", keep="last")
        [["Animal_ID", "Weight_Lbs", "Weight_Date"]]
        .rename(columns={
            "Weight_Lbs": "Latest_Weight",
            "Weight_Date": "Latest_Weight_Date"
        })
    )

    # Add first and latest weights to the performance table
    performance = performance.merge(
        first_weights,
        on="Animal_ID",
        how="left"
    )

    performance = performance.merge(
        latest_weights,
        on="Animal_ID",
        how="left"
    )

else:

    # Create blank columns when no weight records exist
    performance["First_Recorded_Weight"] = np.nan
    performance["First_Weight_Date"] = pd.NaT
    performance["Latest_Weight"] = np.nan
    performance["Latest_Weight_Date"] = pd.NaT


# 4. Determine each animal's initial weight

# If an animal has multiple weight records, use its first recorded weight.
# If it has only one weight record, use Starting_Weight when available.

performance["Initial_Weight"] = (
    performance["Starting_Weight"]
    .combine_first(performance["First_Recorded_Weight"])
)

# When multiple weight records exist, use the first recorded weight
# as the beginning of the tracked period.
multiple_weights = (
    valid_weights.groupby("Animal_ID").size()
    if not valid_weights.empty
    else pd.Series(dtype="int64")
)

performance["Weight_Record_Count"] = (
    performance["Animal_ID"]
    .map(multiple_weights)
    .fillna(0)
)

performance.loc[
    performance["Weight_Record_Count"] >= 2,
    "Initial_Weight"
] = performance.loc[
    performance["Weight_Record_Count"] >= 2,
    "First_Recorded_Weight"
]


# 5. Calculate the number of days tracked

performance["Days_Tracked"] = (
    performance["Latest_Weight_Date"]
    - performance["First_Weight_Date"]
).dt.days

# For animals with only one weight record,
# use Days_On_Feed if available.
performance.loc[
    performance["Weight_Record_Count"] == 1,
    "Days_Tracked"
] = performance.loc[
    performance["Weight_Record_Count"] == 1,
    "Days_On_Feed"
]

# If no weight records exist, leave Days_Tracked blank
performance.loc[
    performance["Weight_Record_Count"] == 0,
    "Days_Tracked"
] = np.nan


# 6. Calculate weight gain

performance["Weight_Gain"] = (
    performance["Latest_Weight"]
    - performance["Initial_Weight"]
)


# 7. Calculate average daily gain (ADG)

performance["ADG"] = (
    performance["Weight_Gain"]
    / performance["Days_Tracked"]
)

# Only calculate ADG when the required information is available
performance.loc[
    (performance["Days_Tracked"] <= 0)
    | (performance["Weight_Gain"].isna())
    | (performance["Initial_Weight"].isna())
    | (performance["Latest_Weight"].isna()),
    "ADG"
] = np.nan


# 8. Compare each animal with the herd average

average_adg = performance["ADG"].mean()

if pd.notna(average_adg):

    performance["Performance"] = np.where(
        performance["ADG"].isna(),
        "Insufficient Weight Data",
        np.where(
            performance["ADG"] > average_adg,
            "Above Herd Average",
            "At or Below Herd Average"
        )
    )

else:

    performance["Performance"] = "Insufficient Weight Data"


# 9. Display results

print("Cattle weight performance calculated!")

print("\nHerd Average Daily Gain:",
      round(average_adg, 2) if pd.notna(average_adg) else "Not available")

display(
    performance[
        [
            "Animal_ID",
            "Breed",
            "Initial_Weight",
            "Latest_Weight",
            "Weight_Gain",
            "Days_Tracked",
            "ADG",
            "Performance"
        ]
    ].round(2)
)

Cattle weight performance calculated!

Herd Average Daily Gain: Not available


,Animal_ID,Breed,Initial_Weight,Latest_Weight,Weight_Gain,Days_Tracked,ADG,Performance


In [36]:
#STEP 5: CALCULATING FEED AND TREATMENT COSTS

# 1. Calculate total allocated feed cost per animal

if not feed_allocations.empty:

    # Keep records with both an Animal ID and allocated cost
    valid_feed = feed_allocations.dropna(
        subset=["Animal_ID", "Allocated_Cost"]
    ).copy()

    # Calculate total feed cost for each animal
    feed_costs = valid_feed.groupby(
        "Animal_ID"
    )["Allocated_Cost"].sum(min_count=1)

else:

    # Allow the Feed Allocations tab to remain empty
    feed_costs = pd.Series(dtype="float64")


# 2. Calculate total treatment cost per animal

if not treatments.empty:

    # Keep records with both an Animal ID and treatment cost
    valid_treatments = treatments.dropna(
        subset=["Animal_ID", "Cost"]
    ).copy()

    # Calculate total treatment cost for each animal
    treatment_costs = valid_treatments.groupby(
        "Animal_ID"
    )["Cost"].sum(min_count=1)

else:

    # Allow the Treatment Records tab to remain empty
    treatment_costs = pd.Series(dtype="float64")


# 3. Add costs to the cattle performance table

performance["Total_Feed_Cost"] = (
    performance["Animal_ID"].map(feed_costs)
)

performance["Total_Treatment_Cost"] = (
    performance["Animal_ID"].map(treatment_costs)
)


# 4. Calculate total recorded costs

# Missing optional costs remain blank instead of becoming $0.

performance["Total_Recorded_Cost"] = performance[
    ["Total_Feed_Cost", "Total_Treatment_Cost"]
].sum(
    axis=1,
    min_count=1
)


# 5. Calculate feed cost per pound of weight gained

performance["Feed_Cost_Per_Lb_Gain"] = (
    performance["Total_Feed_Cost"]
    / performance["Weight_Gain"]
)

# Only calculate this measurement when weight gain is positive
performance.loc[
    (performance["Weight_Gain"] <= 0)
    | (performance["Weight_Gain"].isna())
    | (performance["Total_Feed_Cost"].isna()),
    "Feed_Cost_Per_Lb_Gain"
] = np.nan


# 6. Display cost results

print("Feed and treatment cost analysis completed!")

display(
    performance[
        [
            "Animal_ID",
            "Breed",
            "Weight_Gain",
            "Total_Feed_Cost",
            "Total_Treatment_Cost",
            "Total_Recorded_Cost",
            "Feed_Cost_Per_Lb_Gain"
        ]
    ].round(2)
)


# 7. Show which optional records are available

print("\nCOST RECORD SUMMARY")
print("------------------------------")

print(
    "Animals with feed cost:",
    performance["Total_Feed_Cost"].notna().sum()
)

print(
    "Animals with treatment cost:",
    performance["Total_Treatment_Cost"].notna().sum()
)

print(
    "Animals with recorded costs:",
    performance["Total_Recorded_Cost"].notna().sum()
)

print("\nStep 5 completed successfully!")


Feed and treatment cost analysis completed!


,Animal_ID,Breed,Weight_Gain,Total_Feed_Cost,Total_Treatment_Cost,Total_Recorded_Cost,Feed_Cost_Per_Lb_Gain



COST RECORD SUMMARY
------------------------------
Animals with feed cost: 0
Animals with treatment cost: 0
Animals with recorded costs: 0

Step 5 completed successfully!


In [37]:

# STEP 6: CALCULATING PROCESSING AND SALES INFORMATION
# 1. Prepare sales information

if not sales.empty:

    # Keep records that have an Animal ID
    valid_sales = sales.dropna(
        subset=["Animal_ID"]
    ).copy()

    # Sort by sale date
    valid_sales = valid_sales.sort_values(
        "Sale_Date",
        na_position="first"
    )

    # Keep the latest sale record for each animal
    valid_sales = valid_sales.drop_duplicates(
        subset=["Animal_ID"],
        keep="last"
    )

    # Keep only the information needed for analysis
    sales_summary = valid_sales[
        [
            "Animal_ID",
            "Sale_Date",
            "Final_Live_Weight",
            "Hanging_Weight",
            "Sale_Basis",
            "Price_Per_Lb"
        ]
    ].copy()

    # Standardize sale basis
    sales_summary["Sale_Basis"] = (
        sales_summary["Sale_Basis"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

else:

    # Allow the Sales_Processing tab to remain empty
    sales_summary = pd.DataFrame(
        columns=[
            "Animal_ID",
            "Sale_Date",
            "Final_Live_Weight",
            "Hanging_Weight",
            "Sale_Basis",
            "Price_Per_Lb"
        ]
    )


# 2. Connect sale information to cattle performance

# Remove old sales columns if Step 6 is run more than once
sales_columns = [
    "Sale_Date",
    "Final_Live_Weight",
    "Hanging_Weight",
    "Sale_Basis",
    "Price_Per_Lb"
]

performance = performance.drop(
    columns=sales_columns,
    errors="ignore"
)

performance = performance.merge(
    sales_summary,
    on="Animal_ID",
    how="left"
)


# 3. Calculate dressing percentage

# Dressing percentage compares hanging weight to final live weight.

performance["Dressing_Percentage"] = (
    performance["Hanging_Weight"]
    / performance["Final_Live_Weight"]
) * 100

# Only calculate dressing percentage with valid weights
performance.loc[
    (performance["Final_Live_Weight"] <= 0)
    | (performance["Hanging_Weight"] <= 0)
    | (performance["Final_Live_Weight"].isna())
    | (performance["Hanging_Weight"].isna()),
    "Dressing_Percentage"
] = np.nan


# 4. Determine which weight is used for the sale

performance["Sale_Weight_Used"] = np.nan

# Live-weight sale
live_sale = performance["Sale_Basis"].isin(
    ["live", "live weight", "live-weight"]
)

performance.loc[
    live_sale,
    "Sale_Weight_Used"
] = performance.loc[
    live_sale,
    "Final_Live_Weight"
]

# Hanging-weight sale
hanging_sale = performance["Sale_Basis"].isin(
    ["hanging", "hanging weight", "hanging-weight"]
)

performance.loc[
    hanging_sale,
    "Sale_Weight_Used"
] = performance.loc[
    hanging_sale,
    "Hanging_Weight"
]


# 5. Calculate estimated sale revenue

performance["Estimated_Sale_Revenue"] = (
    performance["Sale_Weight_Used"]
    * performance["Price_Per_Lb"]
)

# Prevent calculations using missing or invalid prices and weights
performance.loc[
    (performance["Sale_Weight_Used"] <= 0)
    | (performance["Price_Per_Lb"] <= 0)
    | (performance["Sale_Weight_Used"].isna())
    | (performance["Price_Per_Lb"].isna()),
    "Estimated_Sale_Revenue"
] = np.nan


# 6. Calculate estimated return after recorded costs

# This is not full profit because other expenses may be missing.

performance["Estimated_Return_After_Recorded_Costs"] = (
    performance["Estimated_Sale_Revenue"]
    - performance["Total_Recorded_Cost"]
)

# Only calculate when both revenue and recorded costs are available
performance.loc[
    performance["Estimated_Sale_Revenue"].isna()
    | performance["Total_Recorded_Cost"].isna(),
    "Estimated_Return_After_Recorded_Costs"
] = np.nan


# 7. Display processing and sales results

print("Processing and sales analysis completed!")

display(
    performance[
        [
            "Animal_ID",
            "Breed",
            "Final_Live_Weight",
            "Hanging_Weight",
            "Dressing_Percentage",
            "Sale_Basis",
            "Price_Per_Lb",
            "Estimated_Sale_Revenue",
            "Estimated_Return_After_Recorded_Costs"
        ]
    ].round(2)
)


# 8. Show which optional sales records are available

print("\nSALES RECORD SUMMARY")
print("------------------------------")

print(
    "Animals with sale records:",
    performance["Sale_Date"].notna().sum()
)

print(
    "Animals with estimated sale revenue:",
    performance["Estimated_Sale_Revenue"].notna().sum()
)

print(
    "Animals with dressing percentage:",
    performance["Dressing_Percentage"].notna().sum()
)

print("\nStep 6 completed successfully!")




Processing and sales analysis completed!


,Animal_ID,Breed,Final_Live_Weight,Hanging_Weight,Dressing_Percentage,Sale_Basis,Price_Per_Lb,Estimated_Sale_Revenue,Estimated_Return_After_Recorded_Costs



SALES RECORD SUMMARY
------------------------------
Animals with sale records: 0
Animals with estimated sale revenue: 0
Animals with dressing percentage: 0

Step 6 completed successfully!


In [39]:
#STEP 7 GENERATING THE HERD PERFORMANCE REPORT
print("=" * 55)
print("          CATTLE HERD PERFORMANCE REPORT")
print("=" * 55)


# 1. Overall herd information

print("\n1. HERD OVERVIEW")
print("-" * 35)

print("Total animals:", len(performance))

print(
    "Animals with weight records:",
    performance["Latest_Weight"].notna().sum()
)

print(
    "Animals with calculated ADG:",
    performance["ADG"].notna().sum()
)

print(
    "Animals with feed cost information:",
    performance["Total_Feed_Cost"].notna().sum()
)

print(
    "Animals with treatment cost information:",
    performance["Total_Treatment_Cost"].notna().sum()
)

print(
    "Animals with sale records:",
    performance["Sale_Date"].notna().sum()
)


# 2. Weight performance summary

print("\n2. WEIGHT PERFORMANCE")
print("-" * 35)

average_gain = performance["Weight_Gain"].mean()
average_adg = performance["ADG"].mean()

print(
    "Average weight gain:",
    round(average_gain, 2)
    if pd.notna(average_gain) else "Not available"
)

print(
    "Average daily gain:",
    round(average_adg, 2)
    if pd.notna(average_adg) else "Not available"
)


# 3. Breed comparison

print("\n3. BREED PERFORMANCE")
print("-" * 35)

breed_summary = (
    performance
    .dropna(subset=["Breed"])
    .groupby("Breed")
    .agg(
        Animals=("Animal_ID", "nunique"),
        Average_Weight_Gain=("Weight_Gain", "mean"),
        Average_ADG=("ADG", "mean")
    )
    .round(2)
)

if not breed_summary.empty:

    display(breed_summary)

else:

    print("No breed information available.")


# 4. Cost summary

print("\n4. COST SUMMARY")
print("-" * 35)

total_allocated_feed = performance[
    "Total_Feed_Cost"
].sum(min_count=1)

total_treatment = performance[
    "Total_Treatment_Cost"
].sum(min_count=1)

total_recorded_cost = performance[
    "Total_Recorded_Cost"
].sum(min_count=1)

print(
    "Allocated feed costs:",
    round(total_allocated_feed, 2)
    if pd.notna(total_allocated_feed) else "Not available"
)

print(
    "Treatment costs:",
    round(total_treatment, 2)
    if pd.notna(total_treatment) else "Not available"
)

print(
    "Total recorded animal costs:",
    round(total_recorded_cost, 2)
    if pd.notna(total_recorded_cost) else "Not available"
)


# 5. Sales and processing summary

print("\n5. SALES AND PROCESSING SUMMARY")
print("-" * 35)

total_revenue = performance[
    "Estimated_Sale_Revenue"
].sum(min_count=1)

average_dressing = performance[
    "Dressing_Percentage"
].mean()

total_estimated_return = performance[
    "Estimated_Return_After_Recorded_Costs"
].sum(min_count=1)

print(
    "Estimated sale revenue:",
    round(total_revenue, 2)
    if pd.notna(total_revenue) else "Not available"
)

print(
    "Average dressing percentage:",
    round(average_dressing, 2)
    if pd.notna(average_dressing) else "Not available"
)

print(
    "Combined estimated return after recorded costs:",
    round(total_estimated_return, 2)
    if pd.notna(total_estimated_return) else "Not available"
)


# 6. Identify the highest-performing animal

print("\n6. TOP-PERFORMING ANIMAL")
print("-" * 35)

animals_with_adg = performance.dropna(
    subset=["ADG"]
)

if not animals_with_adg.empty:

    top_animal = animals_with_adg.loc[
        animals_with_adg["ADG"].idxmax()
    ]

    print("Animal ID:", top_animal["Animal_ID"])
    print("Breed:", top_animal["Breed"])

    print(
        "Weight gain:",
        round(top_animal["Weight_Gain"], 2),
        "lbs"
    )

    print(
        "Average daily gain:",
        round(top_animal["ADG"], 2),
        "lbs/day"
    )

else:

    print("Not enough weight information to identify a top performer.")


# 7. Display the complete animal performance table

print("\n7. COMPLETE ANIMAL PERFORMANCE TABLE")
print("-" * 35)

report_columns = [
    "Animal_ID",
    "Breed",
    "Initial_Weight",
    "Latest_Weight",
    "Weight_Gain",
    "Days_Tracked",
    "ADG",
    "Performance",
    "Total_Feed_Cost",
    "Total_Treatment_Cost",
    "Total_Recorded_Cost",
    "Feed_Cost_Per_Lb_Gain",
    "Final_Live_Weight",
    "Hanging_Weight",
    "Dressing_Percentage",
    "Sale_Basis",
    "Price_Per_Lb",
    "Estimated_Sale_Revenue",
    "Estimated_Return_After_Recorded_Costs"
]

if not performance.empty:

    display(
        performance[report_columns]
        .sort_values(
            "ADG",
            ascending=False,
            na_position="last"
        )
        .round(2)
    )

else:

    print("No cattle records are available for the report.")


# 8. Final report status

print("\n" + "=" * 55)
print("HERD PERFORMANCE REPORT COMPLETED!")
print("=" * 55)


          CATTLE HERD PERFORMANCE REPORT

1. HERD OVERVIEW
-----------------------------------
Total animals: 0
Animals with weight records: 0
Animals with calculated ADG: 0
Animals with feed cost information: 0
Animals with treatment cost information: 0
Animals with sale records: 0

2. WEIGHT PERFORMANCE
-----------------------------------
Average weight gain: Not available
Average daily gain: Not available

3. BREED PERFORMANCE
-----------------------------------
No breed information available.

4. COST SUMMARY
-----------------------------------
Allocated feed costs: Not available
Treatment costs: Not available
Total recorded animal costs: Not available

5. SALES AND PROCESSING SUMMARY
-----------------------------------
Estimated sale revenue: Not available
Average dressing percentage: Not available
Combined estimated return after recorded costs: Not available

6. TOP-PERFORMING ANIMAL
-----------------------------------
Not enough weight information to identify a top performer.

7